In [1]:
import os
import numpy as np
import mne

from scipy.io import loadmat
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from mne.decoding import CSP

In [9]:
# ============================
# CONFIGURATION
# ============================

data_folder = "../data/BCI_IV_2a"
labels_folder = "../data/true_labels"

subjects = [
    "A01", "A02", "A03", "A04", "A05",
    "A06", "A07", "A08", "A09"
]

event_id_train = {
    "LEFT": 7,
    "RIGHT": 8,
    "FOOT": 9,
    "TONGUE": 10
}

# MNE codes → official labels
label_mapping = {
    7: 1,
    8: 2,
    9: 3,
    10: 4
}


In [15]:
# ============================
# FUNCTION: LOAD TRAINING
# ============================

def load_training(subject):

    file_path = os.path.join(
        data_folder,
        subject + "T.gdf"
    )

    raw = mne.io.read_raw_gdf(
        file_path,
        preload=True,
        verbose=False
    )

    # Keep EEG channels
    raw.pick_types(eeg=True)

    # Get events
    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    print(subject, "event dictionary:", event_dict)

    # Find the MNE event IDs corresponding to the
    # original GDF descriptions
    required_descriptions = [
        "769",  # Left
        "770",  # Right
        "771",  # Foot
        "772"   # Tongue
    ]

    available = {}

    for description in required_descriptions:

        if description in event_dict:
            available[description] = event_dict[description]

    print(subject, "motor-imagery events:", available)

    # Make sure all 4 classes exist
    if len(available) != 4:
        raise ValueError(
            f"{subject} does not contain all four motor-imagery events."
        )

    # Create epochs
    epochs = mne.Epochs(
        raw,
        events,
        event_id=available,
        tmin=0,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()

    # Convert event descriptions to official labels
    # 769 -> 1 LEFT
    # 770 -> 2 RIGHT
    # 771 -> 3 FOOT
    # 772 -> 4 TONGUE

    mne_to_label = {
        available["769"]: 1,
        available["770"]: 2,
        available["771"]: 3,
        available["772"]: 4
    }

    y = np.array([
        mne_to_label[event]
        for event in epochs.events[:, -1]
    ])

    print(
        subject,
        "classes:",
        np.unique(y, return_counts=True)
    )

    return X, y

In [16]:
# ============================
# FUNCTION: LOAD TESTING
# ============================

def load_testing(subject):

    file_path = os.path.join(
        data_folder,
        subject + "E.gdf"
    )

    raw = mne.io.read_raw_gdf(
        file_path,
        preload=True,
        verbose=False
    )

    raw.pick_types(eeg=True)

    events, event_dict = mne.events_from_annotations(
        raw,
        verbose=False
    )

    # 783 = evaluation trial marker
    epochs = mne.Epochs(
        raw,
        events,
        event_id={"TRIAL": 7},
        tmin=0,
        tmax=4,
        baseline=None,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()

    # Load official labels
    label_path = os.path.join(
        labels_folder,
        subject + "E.mat"
    )

    mat = loadmat(label_path)

    y = mat["classlabel"].reshape(-1)

    return X, y


In [17]:
# ============================
# FILTER
# ============================

def filter_data(X):

    X_filtered = X.copy()

    for i in range(X_filtered.shape[0]):

        X_filtered[i] = mne.filter.filter_data(
            X_filtered[i],
            sfreq=250,
            l_freq=8,
            h_freq=30,
            verbose=False
        )

    return X_filtered

In [18]:
# ============================
# RUN ALL SUBJECTS
# ============================

results = []

for subject in subjects:

    print("\n==============================")
    print("SUBJECT:", subject)
    print("==============================")

    # Training session
    X_train, y_train = load_training(subject)

    # Testing session
    X_test, y_test = load_testing(subject)

    print("Training:", X_train.shape)
    print("Testing :", X_test.shape)

    # Filter
    X_train = filter_data(X_train)
    X_test = filter_data(X_test)

    # CSP
    csp = CSP(
        n_components=8,
        reg=None,
        log=True,
        norm_trace=False
    )

    X_train_csp = csp.fit_transform(
        X_train,
        y_train
    )

    X_test_csp = csp.transform(
        X_test
    )

    # SVM
    svm = SVC(
        kernel="rbf",
        C=10,
        gamma="scale"
    )

    svm.fit(
        X_train_csp,
        y_train
    )

    # Prediction
    y_pred = svm.predict(
        X_test_csp
    )

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    print(
        "Accuracy:",
        round(accuracy * 100, 2),
        "%"
    )

    print(
        "Macro F1:",
        round(f1, 4)
    )

    results.append({
        "Subject": subject,
        "Accuracy": accuracy,
        "Macro_F1": f1
    })




SUBJECT: A01


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
A01 event dictionary: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
A01 motor-imagery events: {'769': 7, '770': 8, '771': 9, '772': 10}
A01 classes: (array([1, 2, 3, 4]), array([72, 72, 72, 72]))


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Training: (288, 25, 1001)
Testing : (288, 25, 1001)
Computing rank from data with rank=None
    Using tolerance 6.7e-05 (2.2e-16 eps * 25 dim * 1.2e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
Accuracy: 67.71 %
Macro F1: 0.6694

SUBJECT: A02


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Training: (288, 25, 1001)
Testing : (288, 25, 1001)
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 25 dim * 2.2e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
Accuracy: 66.67 %
Macro F1: 0.6738

SUBJECT: A09


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
A09 event dictionary: {np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
A09 motor-imagery events: {'769': 7, '770': 8, '771': 9, '772': 10}
A09 classes: (array([1, 2, 3, 4]), array([72, 72, 72, 72]))


C:\Users\chahi\AppData\Local\Python\pythoncore-3.11-64\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Training: (288, 25, 1001)
Testing : (288, 25, 1001)
Computing rank from data with rank=None
    Using tolerance 0.00012 (2.2e-16 eps * 25 dim * 2.1e+10  max singular value)
    Estimated rank (data): 25
    data: rank 25 computed from 25 data channels with 0 projectors
Reducing data rank from 25 -> 25
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Estimating class=4 covariance using EMPIRICAL
Done.
Accuracy: 67.71 %
Macro F1: 0.6722


In [19]:
# ============================
# FINAL RESULTS
# ============================

print("\n\n==============================")
print("FINAL CROSS-SESSION RESULTS")
print("==============================")

for result in results:

    print(
        result["Subject"],
        "→",
        round(result["Accuracy"] * 100, 2),
        "%",
        "| F1:",
        round(result["Macro_F1"], 4)
    )


accuracies = [
    r["Accuracy"]
    for r in results
]

f1_scores = [
    r["Macro_F1"]
    for r in results
]

print("\nMean Accuracy:",
      round(np.mean(accuracies) * 100, 2),
      "%")

print("Std Accuracy:",
      round(np.std(accuracies) * 100, 2),
      "%")

print("Mean Macro F1:",
      round(np.mean(f1_scores), 4))



FINAL CROSS-SESSION RESULTS
A01 → 67.71 % | F1: 0.6694
A02 → 51.74 % | F1: 0.4868
A03 → 69.79 % | F1: 0.6767
A04 → 61.11 % | F1: 0.6035
A05 → 28.82 % | F1: 0.2581
A06 → 44.44 % | F1: 0.4198
A07 → 52.78 % | F1: 0.4905
A08 → 66.67 % | F1: 0.6738
A09 → 67.71 % | F1: 0.6722

Mean Accuracy: 56.75 %
Std Accuracy: 12.91 %
Mean Macro F1: 0.5501
